<a href="https://colab.research.google.com/github/sotesh1516/deep-ML/blob/main/CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import PIL
import numpy
import IPython
from io import BytesIO
from numpy.lib.stride_tricks import sliding_window_view
images = []
targets = []
for i in range(1,5):
  im = PIL.Image.open(f'/content/drive/MyDrive/Colab Notebooks/{i}.jpg').convert('L')
  bio = BytesIO() #treat a chuck of mem as file, since PIL only writes to file obj
  im.save(bio, format="png")
  display(IPython.display.Image(data=bio.getvalue(), format='png'))
  images.append(numpy.asarray(im)/255) #squash from 0-1
  targets.append(i/4) #squash 0.25 to 1

In [3]:
numpy.random.seed(5814)

In [20]:
#a 5x5 conv, no bias
w1 = numpy.random.randn(5,5)
#a 3x3 conv, no bias. will follow a max pool layer
w2 = numpy.random.randn(3,3)
print(w2)
#finally fcl, will follow another max pool layer
w3 = numpy.random.randn(4,1) # row of fc is pre-determined since input dim, conv_kernel size, max_pool size are hard-coded

[[-0.43239724 -0.90372789 -1.07595349]
 [ 0.50819198  0.48289897  0.00264458]
 [-1.66770468 -0.4490658  -0.22387632]]


In [22]:
num_epoch = 1000
learning_rate = 0

loss_before_training = 0
loss_after_training = 0

final_output, start_output = numpy.array([[]]), numpy.array([[]])

In [6]:
# this won't work if kernel > input
def my_2dconv(X, weights):
  rows, cols = X.shape
  kernel_rows, kernel_cols = weights.shape

  output_rows = rows - kernel_rows + 1
  output_cols = cols - kernel_cols + 1
  output = numpy.zeros((output_rows, output_cols))

  for i in range(output_rows):
    for j in range(output_cols):
      window = X[i:i+kernel_rows,j:j+kernel_cols]
      product = (window * weights).sum()
      output[i, j] = product


  return output

def my_2dconv_backward_input(X, weights, upstream_grad): #return dL/dX
  rows, cols = X.shape
  kernel_rows, kernel_cols = weights.shape

  rotated_kernel = numpy.fliplr(numpy.flipud(weights))

  row_padding = kernel_rows - 1
  col_padding = kernel_cols - 1

  padded_upstream = numpy.pad(upstream_grad, ((row_padding, row_padding),
   (col_padding, col_padding)), mode='constant')


  return my_2dconv(padded_upstream, rotated_kernel)


def my_2dconv_backward_weights(X, weights, upstream_grad): #return dL/dW
  rows, cols = X.shape
  upstream_rows, upstream_cols = upstream_grad.shape
  kernel_rows, kernel_cols = weights.shape
  output = numpy.zeros((kernel_rows, kernel_cols))

  for i in range(kernel_rows):
    for j in range(kernel_cols):

      input_h_start = i #assuming stride = 1
      input_h_end = input_h_start + upstream_rows
      input_w_start = j
      input_w_end = input_w_start + upstream_cols

      current_input_window = X[input_h_start:input_h_end, input_w_start:input_w_end]
      output[i,j] += (current_input_window * upstream_grad).sum()


  return output


def max_pool_2x2(X):
  rows, cols = X.shape

  output_rows = int(((rows - 2)/2)+1)
  output_cols = int(((cols - 2)/2)+1)
  output = numpy.zeros((output_rows, output_cols))

  for i in range(output_rows):
    for j in range(output_cols):
      window = X[i:i+2,j:j+2]
      output[i,j] = numpy.max(window)


  return output

def max_pool_2x2_backward(X, upstream_grad):
  rows, cols = X.shape
  upstream_rows, upstream_cols = upstream_grad.shape
  stride = 2
  pool_size = 2

  output = numpy.zeros((rows, cols))

  for i in range(upstream_rows):
    for j in range(upstream_cols):

      pool_h_start = i * stride
      pool_h_end = pool_h_start + pool_size
      pool_w_start = j * stride
      pool_w_end = pool_w_start + pool_size

      current_input_window = X[pool_h_start:pool_h_end, pool_w_start:pool_w_end]
      mask = (current_input_window == numpy.max(current_input_window))

      output[pool_h_start:pool_h_end, pool_w_start:pool_w_end] += mask * upstream_grad[i,j]


  return output

def relu(X):
  return numpy.maximum(X, 0)

def relu_gradient(X): # no need to deal with any other input since all relu needs is the upstream_grad which in this case is X
  return (X > 0).astype(float)

def sigmoid(X):
  return 1 / (1 + numpy.exp(-X))

def sigmoid_gradient(X):
  return sigmoid(X) * (1 - sigmoid(X))

In [ ]:
print(targets)

[0.25, 0.5, 0.75, 1.0]


In [23]:
for i in range(len(images)):
  conv_1 = my_2dconv(images[i], w1)
  relu_1 = relu(conv_1)
  max_pool_1 = max_pool_2x2(relu_1)
  conv_2 = my_2dconv(max_pool_1, w2)
  max_pool_2 = max_pool_2x2(conv_2)
  max_pool_2_flatten = max_pool_2.flatten()
  fc = max_pool_2_flatten @ w3 # matrix mul, (1 x 4) @ (4 x 1)
  output_from_network = sigmoid(fc)

  # pred = numpy.argmax(output_from_network, axis=1) no need since i change the fc from 4x4 to 4x1
  # (f(x,y) = \sqrt(\sum_i (x_i - y_i)^2)
  error = output_from_network - targets[i]
  loss = numpy.sqrt(numpy.sum(error**2))
  loss_before_training += loss
  start_output = numpy.append(start_output, output_from_network)

In [27]:
for iteration in range(num_epoch):
  for i in range(len(images)):
    conv_1 = my_2dconv(images[i], w1) # 12 x 12
    relu_1 = relu(conv_1)
    max_pool_1 = max_pool_2x2(relu_1) # 6 x 6
    conv_2 = my_2dconv(max_pool_1, w2) # 4 x 4
    max_pool_2 = max_pool_2x2(conv_2) # 2 x 2
    max_pool_2_flatten = max_pool_2.flatten()
    fc = max_pool_2_flatten @ w3 # matrix mul, (1 x 4) @ (4 x 1)
    output_from_network = sigmoid(fc)

    error = output_from_network - targets[i]
    loss = numpy.sqrt(numpy.sum(error**2))
    derv_loss = error / (loss + 1e-8)
    derv_output_from_network = derv_loss * sigmoid_gradient(fc)
    derv_fc_input = numpy.array([derv_output_from_network]) @ w3.T
    x_col = max_pool_2_flatten.reshape(-1, 1)
    derv_fc_weight = x_col * output_from_network #update the weights
    derv_fc_input_spatial = derv_fc_input.reshape(max_pool_2.shape)
    derv_max_pool_2 = max_pool_2x2_backward(conv_2, derv_fc_input_spatial)
    derv_conv_2_input = my_2dconv_backward_input(max_pool_1, w2, derv_max_pool_2)
    derv_conv_2_weight = my_2dconv_backward_weights(max_pool_1, w2, derv_max_pool_2) #update the weights
    derv_max_pool_1 = max_pool_2x2_backward(relu_1, derv_conv_2_input)

    derv_relu = derv_max_pool_1 * relu_gradient(conv_1)
    derv_conv_1_input = my_2dconv_backward_input(images[i], w1, derv_relu)
    derv_conv_1_weight = my_2dconv_backward_weights(images[i], w1, derv_relu)

    #weight updates
    w1 -= learning_rate * derv_conv_1_weight
    w2 -= learning_rate * derv_conv_2_weight
    w3 -= learning_rate * derv_fc_weight




In [28]:
for i in range(len(images)):
  conv_1 = my_2dconv(images[i], w1)
  relu_1 = relu(conv_1)
  max_pool_1 = max_pool_2x2(relu_1)
  conv_2 = my_2dconv(max_pool_1, w2)
  max_pool_2 = max_pool_2x2(conv_2)
  max_pool_2_flatten = max_pool_2.flatten()
  fc = max_pool_2_flatten @ w3 # matrix mul, (1 x 4) @ (4 x 1)
  output_from_network = sigmoid(fc)

  loss =  numpy.sqrt(numpy.sum((output_from_network - targets[i])**2))
  loss_after_training += loss
  final_output = numpy.append(final_output, output_from_network)

In [29]:
print("\nW1 :",w1, "\n\nW2 :", w2, "\n\nW3 :", w3)
print("---------------------------")
print("Loss before Training: ",loss_before_training)
print("Loss after Training: ",loss_after_training)
print("----------------")
print("Start Output : ", start_output)
print("Final Output : ", final_output)
print("Ground Truth  : ", targets)


W1 : [[-0.1887358   1.0339545  -0.67436523  0.81906862 -2.0710027 ]
 [-1.33604438  1.95200011  0.38842331  0.68361945 -1.19165001]
 [ 1.74309941  0.21402119 -1.2398103   1.0176404  -0.57106398]
 [ 1.2040006  -2.33762652  0.30337468  0.14952125  0.13871811]
 [ 0.18431346 -0.42228718 -0.78754294  0.38972621 -1.32351075]] 

W2 : [[-0.43239724 -0.90372789 -1.07595349]
 [ 0.50819198  0.48289897  0.00264458]
 [-1.66770468 -0.4490658  -0.22387632]] 

W3 : [[0.25329493]
 [1.09427578]
 [0.51210542]
 [1.08262218]]
---------------------------
Loss before Training:  1.8681510721063435
Loss after Training:  3.736302144212687
----------------
Start Output :  [1.66336022e-01 2.77314603e-04 1.02907608e-03 4.64206515e-01]
Final Output :  [1.66336022e-01 2.77314603e-04 1.02907608e-03 4.64206515e-01
 1.66336022e-01 2.77314603e-04 1.02907608e-03 4.64206515e-01]
Ground Truth  :  [0.25, 0.5, 0.75, 1.0]
